# <span style="color: #3498db;">**Multi-Objective Grey Wolf Segmentation Optimization (MOGWSO)**</span>  
### <span style="color: #2ecc71;">**Version 1.0.0**</span>  
#### <span style="color: #e74c3c;">**June 2025**</span>  
#### <span style="color: #9b59b6;">**Credit: M. Javad Soltani**</span>  
**Master Student of Remote Sensing and Photogrammetry**  
**K. N. Toosi University of Technology**  

In [ ]:
import os
import pandas as pd
import random
import matplotlib.pyplot as plt
from multiprocessing import Pool, cpu_count

from Mapping_Values import map_params
from Parallelization import process_wolf
from Mirroring import mirror_boundary

def gwo_optimizer(
    base_output_dir,
    reference_path,
    input_img,
    num_wolves=10,
    max_iter=10,
    W1 = 0.33,
    W2 = 0.33,
    K_Factor = 5.403940886699507,
    param_bounds=None,
    log_csv=None,
    plot_output=None
):
    """
    Grey Wolf Optimizer (GWO) for multi-objective image segmentation parameter tuning.
    
    Optimizes segmentation parameters using the Grey Wolf Algorithm,
    balancing accuracy (F1-score, IoU) and over-segmentation penalties.

    Args:
        base_output_dir (str): Directory to save optimization logs/plots.
        reference_path (str): Path to ground truth reference shapefile (.shp).
        input_img (str): Path to input image to segment (.tif).
        num_wolves (int): Number of search agents or wolves (default: 10).
        max_iter (int): Maximum iterations (default: 10).
        W1 (float): Weight for eliminated polygons factor in objective function (default: 0.33).
        W2 (float): Weight for over segmentation factor or OSF (default: 0.33).
        K_Factor (float): The OSF for most Over segmented output. (default: 5.403940886699507)
        param_bounds (dict): The Boundaries of segmentation parameters. Defaults to:
            {
                "num_clusters": [10, 100],
                "min_n_pxls": [5, 40],
                "dist_thres": [10, 150],
                "min_clump_size_factor": [1.2, 1.5],
                "pxl_val_thres_factor": [1.2, 2.0]
            }
        log_csv (str): Custom path for CSV log (default: f"{base_output_dir}/optimization_log.csv").
        plot_output (str): Custom path for progress plot (default: f"{base_output_dir}/gwo_progress.png").

    Returns:
        tuple: (best_params_dict, best_score) where:
            - best_params_dict: Dictionary of optimized parameters.
            - best_score: Highest objective score achieved.

    Example:
        >>> best_params, best_score = gwo_optimizer(
        ...     base_output_dir="results",
        ...     reference_path="gt.shp",
        ...     input_img="input.tif",
        ...     num_wolves=15,
        ...     max_iter=20
        ... )
    """
    os.makedirs(base_output_dir, exist_ok=True)
    log_csv = log_csv or os.path.join(base_output_dir, "optimization_log.csv")
    plot_output = plot_output or os.path.join(base_output_dir, "gwo_progress.png")

    if param_bounds is None:
        param_bounds = {
            "num_clusters": [10, 100],
            "min_n_pxls": [5, 40],
            "dist_thres": [10, 150],
            "min_clump_size_factor": [1.2, 1.5],
            "pxl_val_thres_factor": [1.2, 2.0]
        }

    wolves = [
        {param: random.uniform(0, 1) for param in param_bounds}
        for _ in range(num_wolves)
    ]
    
    log_data = []
    alpha, alpha_score = None, -float("inf")
    beta, beta_score = None, -float("inf")
    delta, delta_score = None, -float("inf")
    a = 2  # Initial 'a' value
    for iteration in range(max_iter):
        print(f"\n Iteration {iteration + 1}/{max_iter}, a = {a:.6f}...")

        iteration_output_dir = os.path.join(base_output_dir, f"iter_{iteration + 1}")
        os.makedirs(iteration_output_dir, exist_ok=True)

        max_processes = min(10, max(1, int(cpu_count() * 0.8))) # prefer to use 10 cores, but if there are less then 12 cores, 80% of them will be used.  
        with Pool(processes=max_processes) as pool:
            results = pool.map(
                process_wolf,
                [
                    (
                        iteration,
                        wolf_idx,
                        wolves[wolf_idx], 
                        param_bounds,
                        input_img,
                        reference_path,
                        W1, 
                        W2,
                        K_Factor,
                        iteration_output_dir,
                    )
                    for wolf_idx in range(num_wolves)
                ]
            )
        # Save Results
        for wolf_idx, (wolf, metrics, objective_score) in enumerate(results):
            if metrics:
                print(f"[ Wolf-{wolf_idx+1}] Metrics Ready for Logging: {objective_score}") 
                print(f"Wolf {wolf_idx + 1}: {wolf}")
            else:
                print(f"[⚠️ Wolf-{wolf_idx+1}] No metrics (possibly empty segmentation).")
            
            
            log_data.append({
                **wolf,
                "iteration": iteration + 1,
                "wolf": wolf_idx + 1,
                "average_f1_score": metrics.get("average_f1_score", "N/A"),
                "num_eliminated_polygons": metrics.get("num_eliminated_polygons", "N/A"),
                "over_segmentation_factor": metrics.get("over_segmentation_factor", "N/A"),
                "weighted_mean_iou": metrics.get("weighted_mean_iou", "N/A"),  
                "mean_iou": metrics.get("mean_iou", "N/A"),                
                "objective_score": objective_score,
            })

        if results:
            # Sort results by objective_score (which is at index 2) in descending order
            sorted_results = sorted(results, key=lambda x: x[2], reverse=True)

            # Assign Alpha, Beta, Delta Wolves based on sorted results
            alpha, _, alpha_score = sorted_results[0]
            beta, _, beta_score = sorted_results[1] if len(sorted_results) > 1 else (alpha, {}, alpha_score)
            delta, _, delta_score = sorted_results[2] if len(sorted_results) > 2 else (beta, {}, beta_score)
        # Log Alpha for Each Iteration (Keep Last Best)
        if alpha is not None:
            log_data.append({
                **alpha,
                "iteration": iteration + 1,
                "wolf": "Alpha",
                "average_f1_score": metrics.get("average_f1_score", None),
                "num_eliminated_polygons": metrics.get("num_eliminated_polygons", None),
                "over_segmentation_factor": metrics.get("over_segmentation_factor", None),
                "weighted_mean_iou": metrics.get("weighted_mean_iou", None),
                "mean_iou": metrics.get("mean_iou", None),
                "objective_score": alpha_score,
            })
        else:
            print(f"⚠️ Warning: No alpha found for iteration {iteration + 1}. Skipping alpha log.")

        print(f"Iteration {iteration + 1}: Alpha Score = {alpha_score:.6f}")
        print(f"Iteration {iteration + 1}: Beta Score = {beta_score:.6f}")
        print(f"Iteration {iteration + 1}: Delta Score = {delta_score:.6f}")

        # Updating Wolfs
        for idx, wolf_position in enumerate(wolves):
            updated_wolf = {}
            for param in wolf_position.keys():
                min_val, max_val = param_bounds[param][:2]
                # print(min_val, max_val)
                r1, r2 = random.random(), random.random()
                A = 2 * a * r1 - a
                C = 2 * r2
                X1 = alpha[param] - A * abs(C * alpha[param] - wolf_position[param])
                X2 = beta[param] - A * abs(C * beta[param] - wolf_position[param])
                X3 = delta[param] - A * abs(C * delta[param] - wolf_position[param])
                new_value = (X1 + X2 + X3) / 3
                # print('New Raw Value: ', new_value)
                # Apply mirror function if new_value is outside boundaries
                updated_wolf[param] = mirror_boundary(new_value, min_val, max_val)

            wolves[idx] = updated_wolf
            # print(f"Updated Value For Wolf {idx + 1}: {updated_wolf}")
            
        #  Rescale Wolves Back to [0,1] for Next Iteration
        for idx, wolf in enumerate(wolves):
            for param in wolf.keys():
                min_val, max_val = param_bounds[param][:2]
                wolf[param] = (wolf[param] - min_val) / (max_val - min_val)  # Rescale

            # print(f" Rescaled Wolf-{idx + 1}: {wolf}")  # Debugging  
        print('Final Wolves:', wolves)  
                
        a = 2 - (2 * (iteration + 1) / max_iter)  # Decrease 'a' linearly by considering the max_iter
    #  9. Save Results to CSV
    log_df = pd.DataFrame(log_data)
    log_df.to_csv(log_csv, index=False)
    print(f" Optimization log saved to {log_csv}")

    #  10. Plot Alpha Score Progress
    alpha_progress = (
        log_df[log_df["wolf"] == "Alpha"]
        .groupby("iteration")["objective_score"]
        .max()
        .cummax()
    )


    plt.figure(figsize=(10, 6))
    plt.plot(alpha_progress, label="Best Objective Score (Alpha)", marker='o', linestyle='--')
    plt.xlabel("Iteration")
    plt.ylabel("Objective Score")   
    plt.title("MOGWO Optimization Progress")
    plt.legend()
    plt.grid(True)
    plt.savefig(plot_output, dpi=300)
    plt.show()

    print("\n✅ Optimization Completed!")
    return alpha, alpha_score


In [ ]:
gwo_optimizer(
        base_output_dir="C:/Users/40223314/Mcs_Thesis/Implimantations/__Grid_Search/13. W1_0.05-W2_0.15_W3_0.8",
        reference_path=r"C:\Users\40223314\Mcs_Thesis\Data\Ref_Data\Final_Ref",
        input_img="C:/Users/40223314/Mcs_Thesis/Data/Input_Seg/Sentinel2_36Band_Composite.tif",
        num_wolves=100,
        max_iter=100, 
        W1 = 0.05, 
        W2 = 0.15)
